--------------------------------------------------------------------------------------------------------------------

**This code uses the following dataset:**

    - Thredd's UCAR ERA5 Reanalysis data on a .25 Degree Latitude/Longitude Grid:
      https://thredds.rda.ucar.edu/thredds/catalog/files/g/d633000/e5.oper.an.pl/202112/catalog.html
  
--------------------------------------------------------------------------------------------------------------------

## ERA5 Temperature and Geopotential Heights

This notebook computes and plots **geopotential height** and **temperature** at a selected pressure level from locally stored ERA5 GRIB files. The purpose of this code is to highlight synoptic-scale weather patterns like ridges, troughs, and thermal gradients across the US.

### Physical Background
**Geopotential Height (Z)** is the height of a constant pressure surface above sea level, contoured in decameters (dam). Troughs and ridges organize the flow and control cold air advection.

**Temperature (K)** at pressure levels identifies thermal boundaries (fronts), cold pools aloft, and warm air advection patterns that can provide insight into heavy precipitation events.

In [ ]:
import os
from pathlib import Path
import shutil
import tempfile

# redirect cfgrib index files to home directory
index_dir = Path("/data1/alamia/.cfgrib_indexes")
index_dir.mkdir(exist_ok=True)
os.environ["CFGRIB_INDEX_PATH"] = str(index_dir)

import xarray as xr
import numpy as np
from datetime import datetime as dt
from metpy.units import units
from metpy import calc as mpcalc
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import cfgrib # noqa: F401 — imported to register the cfgrib xarray backend

## 1.) ERA5 GRIB Data Loader

`get_era5_ds` loads a single pressure-level variable from the read-only ERA5 archive by:

1. Locating the GRIB file at `ERA5_ROOT / variable / variable_YEAR.grib`
2. Copying it to a temporary directory cfgrib can write index files into
3. Opening with `xr.open_dataset(..., engine='cfgrib')`, at the requested level
4. Force-loading into memory with `.load()`, then deleting the temp copy

The `finally` block guarantees cleanup even if the dataset open fails.

In [ ]:
# loads a single pressure-level variable from a local GRIB file because /data1/ERA5 is read-only, each file is copied to a temp directory
# ERA5 local root
era5_root = Path("/data1/ERA5")

def get_era5_ds(variable: str, year: int, level: int):
    filepath = era5_root / variable / f"{variable}_{year}.grib"
    
    if not filepath.exists():
        raise FileNotFoundError(f"No GRIB file found at {filepath}")
    
    # copy to a temp dir you own so cfgrib can write its index file
    tmp_dir = Path(tempfile.mkdtemp(dir="/data1/alamia/.cfgrib_indexes"))
    tmp_file = tmp_dir / filepath.name
    shutil.copy2(filepath, tmp_file)
    
    ds = xr.open_dataset(
        tmp_file,
        engine='cfgrib',
        filter_by_keys={'typeOfLevel': 'isobaricInhPa', 'level': level},
        chunks={'time': 24}
    )
    return ds

## 2.) User Configuration

All run parameters are defined here. No edits should be needed below this cell.

**Extent options:**
- Inner 48 states: `lonW, lonE, latS, latN = -134, -59, 23, 50`
- Including Alaska *(current)*: `lonW, lonE, latS, latN = -180, -30, 0, 70`

**Note:** Coordinates are padded by +-5deg when slicing ERA5 data to avoid edge artifacts in the contour plots near the map boundary.

In [ ]:
# specify vertical level, date/time
vlevel = 850
levelStr = str(vlevel)
Year, Month, Day, Hour, Minute = 2011, 2, 2, 0, 0
dateTime = dt(Year, Month, Day, Hour, Minute)
timeStr = dateTime.strftime("%Y-%m-%d %H%M UTC")

# data dimensions
# lonW, lonE, latS, latN = -134, -59, 23, 50 # for the inner 48 states
lonW, lonE, latS, latN = -180, -30, 0, 70 # to include alaska

# define the folder path
plot_dir = Path("/home1/alamia/Research/ERA5")

## 3.) Load ERA5 Data & Compute Derived Fields

T and Z are loaded at the selected pressure level and at the selected spatial domain. Two derived fields are computed:

- **Geopotential height (dam):** converted from raw geopotential (m²/s²) using MetPy's `geopotential_to_height`, then unit-converted to decameters for standard upper-air display
- **Temperature (T)** — measured in Kelvin

In [ ]:
# Load datasets from local GRIB files at selected pressure level
dsT = get_era5_ds("T", Year, vlevel)
dsZ = get_era5_ds("Z", Year, vlevel)

# subset variables (note: no +360 offset needed, local GRIB uses -180 to 180)
T = dsT['t'].sel(time=dateTime, latitude=slice(latN+5, latS-5), longitude=slice(lonW-5, lonE+5))
Z = dsZ['z'].sel(time=dateTime, latitude=slice(latN+5, latS-5), longitude=slice(lonW-5, lonE+5))

# coordinate arrays
lats = Z.latitude
lons = Z.longitude

# convert geopotential to height (dam)
HGHT = mpcalc.geopotential_to_height(Z).metpy.convert_units('dam')

print(f"Height range: {float(HGHT.min()):.0f}–{float(HGHT.max()):.0f} dam")
print(f"Temperature range: {float(T.min()):.1f}–{float(T.max()):.1f} K")

## 4.) Contour Level Lookup Tables

Geopotential height and wind speed contour intervals vary significantly by pressure level. These lookup tables define `(min, max, interval)` for each supported level, so the correct intervals are selected automatically based on `V_LEVEL`.

Adding a new pressure level requires only a single entry in each dict.

In [ ]:
# geopotential height contour intervals by pressure level (dam)
HGHT_levels = {
    100:  (1560, 1680, 10),
    200:  (1140, 1260, 10),
    250:  (990,  1110, 10),
    300:  (840,  960,  10),
    500:  (504,  600,  6),
    700:  (270,  330,  3),
    850:  (90,   180,  3),
    925:  (15,   90,   3),
}

# temperature contour intervals by pressure level (K)
TEMP_levels = {
    100:  (193, 223, 3),
    200:  (193, 223, 3),
    250:  (203, 233, 3),
    300:  (213, 243, 3),
    500:  (233, 273, 3),
    700:  (243, 293, 3),
    850:  (253, 303, 3),
    925:  (258, 308, 3),
}

# unpack intervals for the selected level
minVal, maxVal, cint = HGHT_levels[vlevel]
minTemp, maxTemp, cint = TEMP_levels[vlevel]

HGHTcintervals = np.arange(minVal, maxVal, cint)
TEMPcintervals = np.arange(minTemp, maxTemp, cint)

## 5.) Temperature & Geopotential Height Plot

Two overlapping layers are plotted:

1. **Temperature filled contours** (`RdYlBu_r`): highlights thermal gradients and frontal boundaries at the selected level
2. **Geopotential height contours** (black): troughs and ridges that organize upper-level flow (decameters)

**Note:** The 850 hPa level is useful for Midwest winter weather as it sits just above the boundary layer and captures low-level jets, warm/cold air advection, and the thermal structure of approaching systems

In [ ]:
# set up map projections
proj_map = ccrs.LambertConformal(central_longitude=-100, central_latitude=30)
proj_data = ccrs.PlateCarree()  # Lat-lon data projection

# PLOT
fig, ax = plt.subplots(subplot_kw={'projection': proj_map}, figsize=(12, 8))

# set extent
#ax.set_extent([-127, -67, 23, 50], crs=ccrs.PlateCarree()) # for the inner 48 states
ax.set_extent([-160, -67, 25, 60], crs=ccrs.PlateCarree()) # to include alaska
# ax.set_extent([-105, -80, 35, 50], crs=ccrs.PlateCarree()) # midwest

# political/geographic boundaries
ax.coastlines(resolution='50m', linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linestyle='--', linewidth=0.5)
ax.add_feature(cfeature.STATES, linewidth=0.5, edgecolor='black')

# lon/lat gridlines
gridlines = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.7, linestyle='--')
gridlines.top_labels   = True
gridlines.right_labels = False
gridlines.xlabel_style = {'size': 12, 'color': 'black'}
gridlines.ylabel_style = {'size': 15, 'color': 'black'}

# 1. temp contour plot
contourf = ax.contourf(
    lons, lats, T, 
    levels=TEMPcintervals, 
    cmap='RdYlBu_r',
    transform=proj_data, 
    extend='both', zorder=1 # extend='both' because data may exceeds fixed level bounds
)

# 2. geopotential height contours and labels
contour = ax.contour(
    lons, lats, HGHT, 
    levels=HGHTcintervals, 
    colors='black', linewidths=.8, 
    linestyles='solid', 
    transform=proj_data, zorder=2
)
ax.clabel(contour, inline=True, fmt='%d', fontsize=10, colors='black', inline_spacing=3)

# color bar (temperature)
cbar = plt.colorbar(contourf, ax=ax, orientation='vertical', 
                    pad=0.04, aspect=30, shrink=0.8, extendrect=True)
cbar.set_label('Temperature (K)', fontsize=12)
cbar.ax.tick_params(labelsize=10)

# title
title = f"ERA-5 {levelStr} hPa Temperature (K) and Geopotential Heights (dam)"
subtitle = f"Valid at: {timeStr}"
plt.title(f"{title}\n{subtitle}", fontsize=16, fontweight='bold', loc='center')

# save and show
plt.tight_layout()

plot_path = plot_dir / f"Temps_({timeStr}).png"
plt.savefig(plot_path, dpi=100)
print(f"Plot saved at: {plot_path}")

plt.show()